# Phase 1: Why Vector-Based RAG Fails at Multi-Hop Reasoning

This notebook demonstrates, with a production-grade workflow, why traditional vector-based Retrieval-Augmented Generation (RAG) systems struggle with multi-hop reasoning in complex documents. We use a simulated 5-page corporate ownership document, chunk it, store it in ChromaDB, and build a basic RAG pipeline. We then attack the system with a multi-hop query and analyze the failure points.

## 1. Import Required Libraries

We import all necessary libraries for document processing, embedding, vector storage, and RAG pipeline construction. Environment variables for API keys are loaded securely.

In [6]:
# Production-Grade Imports and Setup
import os
import sys
import logging
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
import tiktoken

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables securely
load_dotenv()

# Validate API keys
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not (OPENAI_API_KEY or OPENROUTER_API_KEY):
    logger.error("No valid LLM API key found. Please set OPENAI_API_KEY or OPENROUTER_API_KEY in your .env file.")
    sys.exit(1)

# Choose embedding provider (OpenRouter preferred, fallback to OpenAI)
EMBEDDING_PROVIDER = 'openrouter' if OPENROUTER_API_KEY else 'openai'

# Utility: Tokenizer for chunking diagnostics
def count_tokens(text: str, model: str = "gpt-5-mini") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

## 2. Load and Inspect the Sample Document

We simulate a 5-page document describing a complex corporate ownership structure, including multiple shell companies and executive relationships. Each page is represented as a string for demonstration.

In [12]:
# Simulate a 5-page corporate document with rich structure and metadata
from langchain_core.documents import Document
document_pages = [
    {
        "page": 1,
        "text": """
Acme Holdings Ltd. owns 100% of Shell Alpha LLC.
John Smith previously attended a conference hosted by Acme Holdings Ltd.
Acme Holdings Ltd. is registered in the Cayman Islands.
"""
    },
    {
        "page": 2,
        "text": """
Shell Alpha LLC owns 80% of Beta Investments Inc.
The remaining 20% is owned by Gamma Trust.
Jane Doe is CFO of Beta Investments Inc.
Shell Alpha LLC is registered in Delaware.
"""
    },
    {
        "page": 3,
        "text": """
Beta Investments Inc. owns 60% of Delta Trading Corp.
John Smith is also a board member at Delta Trading Corp.
Delta Trading Corp. is registered in Panama.
"""
    },
    {
        "page": 4,
        "text": """
Delta Trading Corp. is a supplier to Omega Manufacturing Ltd.
Omega Manufacturing Ltd. is managed by Emily Clark.
Omega Manufacturing Ltd. is registered in Singapore.
"""
    },
    {
        "page": 5,
        "text": """
John Smith works for Delta Trading Corp.
Delta Trading Corp. is currently under investigation for financial irregularities.
Multiple shell companies are involved in Delta Trading Corp.'s ownership structure.
"""
    }
]

# Display a summary of each page with metadata
documents = [
    Document(
        page_content=p["text"],
        metadata={"page": p["page"]}
    )
    for p in document_pages
]

df = pd.DataFrame(document_pages)
df["preview"] = df["text"].apply(lambda x: " ".join(x.strip().split("\n")[:2]))
display(df[["page", "preview"]])

,page,preview
0,1,Acme Holdings Ltd. owns 100% of Shell Alpha LL...
1,2,Shell Alpha LLC owns 80% of Beta Investments I...
2,3,Beta Investments Inc. owns 60% of Delta Tradin...
3,4,Delta Trading Corp. is a supplier to Omega Man...
4,5,John Smith works for Delta Trading Corp. Delta...


## 3. Chunk the Document for Vector Storage

We split the document into overlapping chunks using a robust strategy. This simulates how production RAG systems preprocess documents for vector storage. We display chunk statistics and inspect a few chunks.

In [13]:

chunk_size = 120
chunk_overlap = 20

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=lambda x: count_tokens(x),
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

chunk_tokens = [count_tokens(doc.page_content) for doc in chunks]

logger.info(f"Total chunks created: {len(chunks)}")
logger.info(
    f"Chunk token stats -> Min: {min(chunk_tokens)}, "
    f"Mean: {np.mean(chunk_tokens):.2f}, "
    f"Max: {max(chunk_tokens)}"
)

2026-05-07 13:01:33,619 INFO Total chunks created: 5
2026-05-07 13:01:33,619 INFO Chunk token stats -> Min: 31, Mean: 35.80, Max: 40


In [14]:

print("CHUNK FRAGMENTATION ANALYSIS")

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(f"Metadata: {chunk.metadata}")
    print("-" * 40)
    print(chunk.page_content.strip())

CHUNK FRAGMENTATION ANALYSIS

Chunk 1
Metadata: {'page': 1}
----------------------------------------
Acme Holdings Ltd. owns 100% of Shell Alpha LLC.
John Smith previously attended a conference hosted by Acme Holdings Ltd.
Acme Holdings Ltd. is registered in the Cayman Islands.

Chunk 2
Metadata: {'page': 2}
----------------------------------------
Shell Alpha LLC owns 80% of Beta Investments Inc.
The remaining 20% is owned by Gamma Trust.
Jane Doe is CFO of Beta Investments Inc.
Shell Alpha LLC is registered in Delaware.

Chunk 3
Metadata: {'page': 3}
----------------------------------------
Beta Investments Inc. owns 60% of Delta Trading Corp.
John Smith is also a board member at Delta Trading Corp.
Delta Trading Corp. is registered in Panama.

Chunk 4
Metadata: {'page': 4}
----------------------------------------
Delta Trading Corp. is a supplier to Omega Manufacturing Ltd.
Omega Manufacturing Ltd. is managed by Emily Clark.
Omega Manufacturing Ltd. is registered in Singapore.

Chun

## 4. Store Chunks in ChromaDB Vector Store

We embed each chunk using a lightweight embedding model and store the embeddings in a ChromaDB vector store. We verify storage by listing stored chunks and their metadata.

In [15]:
# Embed and store chunks in ChromaDB with robust error handling
persist_directory = 'chromadb_store'

if EMBEDDING_PROVIDER == 'openrouter':
    # Placeholder: Replace with actual OpenRouter embedding logic if available
    embeddings = OpenAIEmbeddings(openai_api_key=OPENROUTER_API_KEY)
else:
    embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

try:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    logger.info(f"Stored {vectorstore._collection.count()} chunks in ChromaDB.")
    sample = vectorstore._collection.get(limit=2)
    print("Sample stored metadata:", sample["metadatas"])
except Exception as e:
    logger.error(f"Failed to store chunks in ChromaDB: {e}")
    raise

2026-05-07 13:02:45,747 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-05-07 13:02:45,877 INFO Stored 5 chunks in ChromaDB.


Sample stored metadata: [{'page': 1}, {'page': 2}]


## 5. Implement Basic RAG Pipeline with Semantic Search

We build a simple RAG pipeline: accept a user query, embed it, perform semantic search over the vector store, and return the top-k most relevant chunks.

In [16]:
def inspect_retrieval(query: str, k: int = 4):
    print("=" * 80)
    print("RETRIEVAL ANALYSIS")
    print("=" * 80)

    results = vectorstore.similarity_search_with_score(query, k=k)

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\nRank #{rank}")
        print(f"Similarity Score: {score:.4f}")
        print(f"Page: {doc.metadata.get('page')}")
        print("-" * 40)
        print(doc.page_content.strip())

    return results

In [17]:
# Build a robust RAG pipeline with semantic search and error handling
def build_rag_chain(llm, vectorstore, k=4):
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )

# Use a lightweight LLM for demonstration (OpenAI or OpenRouter)
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, temperature=0)
rag_chain = build_rag_chain(llm, vectorstore, k=4)

def query_rag(query: str, k: int = 4) -> Dict[str, Any]:
    try:
        result = rag_chain({"query": query})
        print("Answer:", result["result"])
        print("\nTop retrieved chunks:")
        for i, doc in enumerate(result["source_documents"]):
            print(f"Chunk {i+1} (tokens: {count_tokens(doc.page_content)}):\n{doc.page_content}\nMeta: {doc.metadata}\n---")
        return result
    except Exception as e:
        logger.error(f"RAG query failed: {e}")
        return {}

## 6. Query the RAG System for Multi-Hop Reasoning

We pose a multi-hop query that requires combining information from non-adjacent pages:

**Query:** Who is the ultimate parent company of the organization John Smith works for?

We run the query through the RAG pipeline and display the retrieved chunks and the system's answer.

In [18]:
# Multi-hop query: requires combining facts from multiple pages
multi_hop_query = """
Which Cayman Islands company indirectly controls
the company where John Smith works?
"""
rag_result = inspect_retrieval(multi_hop_query)

RETRIEVAL ANALYSIS


2026-05-07 13:03:45,842 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Rank #1
Similarity Score: 0.3204
Page: 5
----------------------------------------
John Smith works for Delta Trading Corp.
Delta Trading Corp. is currently under investigation for financial irregularities.
Multiple shell companies are involved in Delta Trading Corp.'s ownership structure.

Rank #2
Similarity Score: 0.3371
Page: 1
----------------------------------------
Acme Holdings Ltd. owns 100% of Shell Alpha LLC.
John Smith previously attended a conference hosted by Acme Holdings Ltd.
Acme Holdings Ltd. is registered in the Cayman Islands.

Rank #3
Similarity Score: 0.3668
Page: 3
----------------------------------------
Beta Investments Inc. owns 60% of Delta Trading Corp.
John Smith is also a board member at Delta Trading Corp.
Delta Trading Corp. is registered in Panama.

Rank #4
Similarity Score: 0.4300
Page: 2
----------------------------------------
Shell Alpha LLC owns 80% of Beta Investments Inc.
The remaining 20% is owned by Gamma Trust.
Jane Doe is CFO of Beta Investmen

## 7. Analyze Retrieval Results and Failure Points

We analyze which chunks were retrieved, which relevant facts were missed (especially those from non-adjacent pages), and why the system failed to answer the multi-hop question. We inspect chunk metadata and retrieval scores.

In [19]:
required_pages = {1, 2, 3, 5}

retrieved_pages = {
    doc.metadata["page"]
    for doc, _ in vectorstore.similarity_search_with_score(
        multi_hop_query,
        k=4
    )
}

print("\nRequired Pages:", required_pages)
print("Retrieved Pages:", retrieved_pages)

missing_pages = required_pages - retrieved_pages

if missing_pages:
    print(f"\nFAILED: Missing critical pages -> {missing_pages}")
else:
    print("\nSUCCESS: All required pages retrieved")

2026-05-07 13:03:50,329 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Required Pages: {1, 2, 3, 5}
Retrieved Pages: {1, 2, 3, 5}

SUCCESS: All required pages retrieved


In [22]:
# Analyze which chunks were retrieved and which were missed

if rag_result:

    retrieved_chunks = [
        doc.page_content
        for doc, score in rag_result
    ]

    # Key facts needed for multi-hop reasoning
    key_facts = [
        "Acme Holdings Ltd. owns 100% of Shell Alpha LLC.",
        "Shell Alpha LLC owns 80% of Beta Investments Inc.",
        "Beta Investments Inc. owns 60% of Delta Trading Corp.",
        "John Smith works for Delta Trading Corp."
    ]

    print("\n" + "=" * 80)
    print("MULTI-HOP FACT COVERAGE ANALYSIS")
    print("=" * 80)

    for i, fact in enumerate(key_facts, 1):

        found = any(
            fact in chunk
            for chunk in retrieved_chunks
        )

        print(
            f"Key Fact {i}: "
            f"{'FOUND' if found else 'NOT FOUND'}"
        )

else:
    print("No retrieval results available.")


MULTI-HOP FACT COVERAGE ANALYSIS
Key Fact 1: FOUND
Key Fact 2: FOUND
Key Fact 3: FOUND
Key Fact 4: FOUND


In [21]:
for k in [1, 2, 3, 4, 5]:
    results = vectorstore.similarity_search_with_score(
        multi_hop_query,
        k=k
    )

    pages = {doc.metadata["page"] for doc, _ in results}

    print(f"k={k} -> Retrieved pages: {pages}")

2026-05-07 13:04:39,093 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


k=1 -> Retrieved pages: {5}


2026-05-07 13:04:40,158 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


k=2 -> Retrieved pages: {1, 5}


2026-05-07 13:04:40,659 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-05-07 13:04:40,852 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


k=3 -> Retrieved pages: {1, 3, 5}
k=4 -> Retrieved pages: {1, 2, 3, 5}


2026-05-07 13:04:41,388 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


k=5 -> Retrieved pages: {1, 2, 3, 4, 5}


## 8. Reflect on Multi-Hop Blindspot in Vector RAG

### Why did the retriever fail to fetch the chunk from page 1?

Traditional vector-based retrievers rely on semantic similarity between the query and individual chunks. If the query does not closely match the wording or context of a chunk (e.g., the chunk from page 1 stating the ultimate parent company), it may not be retrieved, especially if the answer requires combining information from multiple, non-adjacent chunks.

### Why does chunking break multi-hop reasoning?

Chunking fragments the document, breaking relationships that span across chunks. Embeddings capture local context but not the global relationships needed for multi-hop reasoning. As a result, the retriever cannot "connect the dots" between facts scattered across different chunks, leading to failure on complex queries that require reasoning over multiple hops.

### Limitations of Embeddings and Loss of Relationships

Embeddings are powerful for capturing semantic similarity but are inherently limited in representing explicit relationships and multi-step reasoning paths. This leads to a loss of relational information when documents are chunked, making it difficult for vector-based RAG systems to answer multi-hop questions accurately.